# Market Data Module V2

Produces a sorted list of valid price tiers per `(product_id, region)` by combining three market data sources.

## Sources
1. **Ben Soliman** — reference distributor prices (all regions)
2. **Marketplace** — live shelf prices from MaxAB marketplace sellers (regional, with fallback)
3. **Scraped** — competitor prices from Cartona, Tawfeer, Talabia (regional, with fallback; Speed = Alex only)

## Pipeline
Collect prices → regional fallback → floor filter (>= 0.9 x WAC) → commercial group union → step subdivision → output

## Usage
```python
%run market_data_module_2.ipynb

# V2 output: sorted price tier lists for pricing decisions
df_market_v2 = get_market_data_v2()
df_market_cohorts = expand_to_cohorts(df_market_v2)

# Legacy output: percentile columns for DB storage (same as V1)
df_market_legacy = get_market_data_legacy()

# Other functions (from V1, available after %run):
# get_margin_tiers(), get_brand_market_percentiles(),
# fill_brand_market_fallback(), get_market_signals()
```

In [ ]:
# =============================================================================
# LOAD V1 MODULE (provides all legacy functions + queries + setup)
# This gives us: get_market_data(), get_margin_tiers(), get_brand_market_percentiles(),
# fill_brand_market_fallback(), get_market_signals(), query_snowflake, TIMEZONE, etc.
# =============================================================================
import os as _os
if _os.path.exists('market_data_module.ipynb'):
    _v1_path = 'market_data_module.ipynb'
elif _os.path.exists('modules/market_data_module.ipynb'):
    _v1_path = 'modules/market_data_module.ipynb'
else:
    raise FileNotFoundError('Cannot find market_data_module.ipynb')
%run $_v1_path

# Load queries_module for commercial functions
import os as _os2
_qm_path = 'queries_module.ipynb' if _os2.path.exists('queries_module.ipynb') else 'modules/queries_module.ipynb'
if _os2.path.exists(_qm_path):
    %run $_qm_path

get_market_data_legacy = get_market_data

import pandas as pd
import numpy as np
from datetime import datetime
import pytz

ALL_REGIONS = ['Cairo', 'Giza', 'Alexandria', 'Delta East', 'Delta West', 'Upper Egypt']

REGIONAL_FALLBACK = {
    'Cairo':       ['Giza'],
    'Giza':        ['Cairo'],
    'Delta East':  ['Delta West'],
    'Delta West':  ['Delta East'],
    'Alexandria':  ['Delta East', 'Delta West', 'Cairo', 'Giza'],
    'Upper Egypt': ['Cairo', 'Giza'],
}

REGION_COHORT_MAP = {
    'Cairo': [700], 'Giza': [701], 'Alexandria': [702],
    'Delta West': [703], 'Delta East': [704],
    'Upper Egypt': [1123, 1124, 1125, 1126],
}

DEFAULT_TARGET_MARGIN = 0.10
MAX_MARGIN_GAP_PCT = 0.30

print(f'\nMarket Data Module V2 ready')
print('Legacy: get_market_data_legacy(), get_margin_tiers(), get_brand_market_percentiles(), fill_brand_market_fallback(), get_market_signals()')
print('V2: get_market_data_v2(), expand_to_cohorts()')

In [ ]:
# =============================================================================
# SOURCE 1: BEN SOLIMAN
# Two-track logic: main (diff < 30%) and lower (diff > 30% with PU correction)
# Not regional - one price per product, applied to all regions
# =============================================================================
V2_BEN_QUERY = f'''
WITH lower_raw AS (
    SELECT
        sm.maxab_product_id AS product_id,
        sm.injection_date,
        sm.bs_price,
        f.wac1,
        f.wac_p,
        sm.maxab_basic_unit_count,
        cpc.price AS cu_price,
        pup.child_quantity,
        ABS(sm.bs_price - (f.wac_p * sm.maxab_basic_unit_count)) / NULLIF(f.wac_p * sm.maxab_basic_unit_count, 0) AS diff,
        ROUND(cpc.price / sm.bs_price) AS p1
    FROM materialized_views.savvy_mapping sm
    JOIN finance.all_cogs f
        ON f.product_id = sm.maxab_product_id
        AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
    JOIN packing_unit_products pu
        ON pu.product_id = sm.maxab_product_id AND pu.is_basic_unit = 1
    JOIN cohort_product_packing_units cpc
        ON cpc.product_packing_unit_id = pu.id AND cpc.cohort_id = 700
    JOIN packing_unit_products pup
        ON pup.product_id = sm.maxab_product_id AND pup.is_basic_unit = 1
    WHERE sm.bs_price IS NOT NULL
        AND sm.injection_date::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 5
),

-- Main track: BS price close to our cost (diff < 30%%)
main_track AS (
    SELECT
        product_id,
        AVG(bs_price / maxab_basic_unit_count) AS ben_soliman_price,
        MAX(injection_date) AS injection_date
    FROM (
        SELECT *,
            bs_price / maxab_basic_unit_count AS unit_price,
            (bs_price / maxab_basic_unit_count - wac_p) / NULLIF(wac_p, 0) AS margin_diff,
            ROW_NUMBER() OVER (PARTITION BY product_id, bs_price ORDER BY diff) AS rnk
        FROM lower_raw
        WHERE diff < 0.3
        QUALIFY MAX(injection_date::DATE) OVER (PARTITION BY product_id) = injection_date::DATE
    )
    WHERE rnk = 1
        AND margin_diff BETWEEN -0.5 AND 0.5
    GROUP BY product_id
    HAVING ben_soliman_price BETWEEN MIN(wac_p) * 0.8 AND MIN(wac_p) * 1.3
),

-- Lower track: BS price far from cost (diff > 30%%), needs PU correction + WAC sanity check
lower_track AS (
    SELECT
        product_id,
        AVG(adjusted_price) AS ben_soliman_price,
        MAX(injection_date) AS injection_date
    FROM (
        SELECT
            product_id,
            injection_date,
            wac_p,
            bs_price * multiplier AS adjusted_price
        FROM (
            SELECT *,
                CASE WHEN p1 > 1 THEN child_quantity ELSE 0 END AS scheck,
                CASE
                    WHEN (ROUND(p1 / NULLIF(CASE WHEN p1 > 1 THEN child_quantity ELSE 0 END, 0))
                          * (CASE WHEN p1 > 1 THEN child_quantity ELSE 0 END)) = 0
                    THEN ROUND(p1 / 2) * 2
                    ELSE ROUND(p1 / NULLIF(CASE WHEN p1 > 1 THEN child_quantity ELSE 0 END, 0))
                         * (CASE WHEN p1 > 1 THEN child_quantity ELSE 0 END)
                END AS multiplier
            FROM lower_raw
            WHERE diff > 0.3 AND p1 > 1
            QUALIFY MAX(injection_date::DATE) OVER (PARTITION BY product_id) = injection_date::DATE
        )
    )
    WHERE adjusted_price BETWEEN wac_p * 0.9 AND wac_p * 1.2
    GROUP BY product_id
),

-- Combine: main track preferred (prio 1), lower track fallback (prio 2)
combined AS (
    SELECT product_id, ben_soliman_price, injection_date, 1 AS prio FROM main_track
    UNION ALL
    SELECT product_id, ben_soliman_price, injection_date, 2 AS prio FROM lower_track
),

best AS (
    SELECT *
    FROM combined
    QUALIFY MAX(injection_date) OVER (PARTITION BY product_id) = injection_date
        AND prio = MIN(prio) OVER (PARTITION BY product_id)
)

SELECT product_id, AVG(ben_soliman_price) AS price
FROM best
GROUP BY product_id
'''

print('Source 1: Ben Soliman query defined')

In [ ]:
# =============================================================================
# SOURCE 2: MARKETPLACE
# Live shelf prices from MaxAB marketplace sellers, per region
# Cleaning: WAC4 +/-40% filter, IQR on ticket size and max_per_order
# Output: per-basic-unit price per (product, region)
# =============================================================================
V2_MARKETPLACE_QUERY = f'''
WITH seller_region AS (
    SELECT
        seller_retailer.retailer_id,
        CASE WHEN regions.name_en = 'Greater Cairo' THEN cities.name_en
             ELSE regions.name_en END AS region,
        seller_id
    FROM materialized_views.sellers_retailers_mapping seller_retailer
    JOIN retailers ON retailers.id = seller_retailer.retailer_id
    JOIN materialized_views.retailer_polygon ON materialized_views.retailer_polygon.retailer_id = retailers.id
    JOIN districts ON districts.id = materialized_views.retailer_polygon.district_id
    JOIN cities ON cities.id = districts.city_id
    JOIN states ON states.id = cities.state_id
    JOIN regions ON regions.id = states.region_id
    JOIN egypt_marketplace.sellers ON sellers.id = seller_retailer.seller_id AND sellers.status = 'ACTIVE'
),

recent_price AS (
    SELECT wp.product_id, wp.packing_unit_id AS product_pu, wp.price,
           wp.max_per_order, warehouses.seller_id, warehouses.min_ticket_size
    FROM egypt_marketplace.warehouse_products wp
    LEFT JOIN egypt_marketplace.warehouses ON warehouses.id = wp.warehouse_id
    WHERE wp.available > 0 AND wp.total_stock > 0 AND activation = 'true'
),

mp_raw AS (
    SELECT DISTINCT sr.region, rp.product_id, rp.price,
           rp.max_per_order, rp.seller_id, rp.min_ticket_size
    FROM recent_price rp
    JOIN seller_region sr ON sr.seller_id = rp.seller_id
),

pu_wacs AS (
    SELECT pup.product_id, pup.packing_unit_id AS pu_id,
           pup.basic_unit_count AS buc, f.wac4,
           pup.basic_unit_count * f.wac4 AS pu_wac4
    FROM packing_unit_products pup
    LEFT JOIN finance.all_cogs f ON f.product_id = pup.product_id
        AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
),

mp_filtered AS (
    SELECT m.region, m.product_id, m.price / pw.buc AS unit_price,
           m.max_per_order, m.min_ticket_size
    FROM mp_raw m
    JOIN pu_wacs pw ON pw.product_id = m.product_id
    WHERE pw.pu_wac4 > 0
        AND ROUND((m.price - pw.pu_wac4) / pw.pu_wac4 * 100, 2) BETWEEN -40 AND 40
),

ticket_iqr AS (
    SELECT *,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY min_ticket_size)
            OVER (PARTITION BY region, product_id) AS ts_q1,
        PERCENTILE_CONT(0.85) WITHIN GROUP (ORDER BY min_ticket_size)
            OVER (PARTITION BY region, product_id) AS ts_q3
    FROM mp_filtered
),
ticket_cleaned AS (
    SELECT * FROM ticket_iqr
    WHERE min_ticket_size >= ts_q1 - 1.5 * (ts_q3 - ts_q1)
      AND min_ticket_size <= ts_q3 + 3.0 * (ts_q3 - ts_q1)
),

order_iqr AS (
    SELECT *,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY max_per_order)
            OVER (PARTITION BY region, product_id) AS mo_q1,
        PERCENTILE_CONT(0.85) WITHIN GROUP (ORDER BY max_per_order)
            OVER (PARTITION BY region, product_id) AS mo_q3
    FROM ticket_cleaned
),
cleaned AS (
    SELECT region, product_id, unit_price AS price
    FROM order_iqr
    WHERE max_per_order >= mo_q1 - 1.5 * (mo_q3 - mo_q1)
)

SELECT region, product_id, price, 'marketplace' AS source
FROM cleaned
'''

print('Source 2: Marketplace query defined')

In [ ]:
# =============================================================================
# SOURCE 3: SCRAPED
# Competitor prices: Cartona, Tawfeer, Talabia (Speed = Alexandria only)
# Cartona: keep where min_ticket_size < 8000 AND quantity_per_unit > 5
# Match to closest PU by WAC (within 30% diff)
# =============================================================================
V2_SCRAPED_QUERY = f'''
WITH latest_per_sku AS (
    SELECT product_name_ar, MAX(created_at::DATE) AS max_date
    FROM materialized_views.raw_scraped_data
    WHERE created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 4
    GROUP BY product_name_ar
),

main_prices AS (
    SELECT DISTINCT r.*
    FROM materialized_views.raw_scraped_data r
    JOIN latest_per_sku l ON r.product_name_ar = l.product_name_ar AND r.created_at::DATE = l.max_date
    WHERE (
        (r.source_app = 'Speed' AND COALESCE(r.region, r.area) IN ('Alex', 'Alexandria'))
        OR r.source_app IN ('Cartona', 'Tawfeer', 'Talabia')
    )
),

cleaned_data AS (
    SELECT cm.query_id AS product_id,
           CASE COALESCE(mp.region, mp.area)
               WHEN 'Giza' THEN 'Cairo'
               WHEN 'Alex' THEN 'Alexandria'
               ELSE COALESCE(mp.region, mp.area)
           END AS region,
           mp.price AS comp_price,
           mp.source_app
    FROM main_prices mp
    JOIN materialized_views.competitors_mapping_fixed cm ON cm.match = mp.product_name_ar
),

matched_prices AS (
    SELECT cd.product_id, cd.region,
           cd.comp_price / pup.basic_unit_count AS price,
           cd.source_app AS source
    FROM cleaned_data cd
    JOIN finance.all_cogs f ON f.product_id = cd.product_id
        AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
    JOIN packing_unit_products pup ON pup.product_id = cd.product_id
    WHERE ABS(cd.comp_price - f.wac1 * pup.basic_unit_count) / NULLIF(f.wac1 * pup.basic_unit_count, 0) < 0.3
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY cd.product_id, COALESCE(cd.source_app, 'unknown'), cd.region
        ORDER BY ABS(cd.comp_price - f.wac1 * pup.basic_unit_count) / NULLIF(f.wac1 * pup.basic_unit_count, 0)
    ) = 1
)

SELECT region, product_id, price, source
FROM matched_prices
'''

print('Source 3: Scraped query defined')

In [ ]:
# =============================================================================
# SUPPORTING QUERIES
# =============================================================================

V2_WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN from_date AND to_date
'''

V2_TARGET_MARGINS_QUERY = f'''
WITH brand_cat_target AS (
    SELECT DISTINCT
        b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_bm
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    QUALIFY CASE
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month')
    END = DATE_TRUNC('month', date)
),
cat_target AS (
    SELECT cat,
           SUM(target_bm * (target_nmv / NULLIF(cat_total, 0))) AS cat_target_margin
    FROM (
        SELECT *, SUM(target_nmv) OVER (PARTITION BY cat) AS cat_total
        FROM (
            SELECT cat, brand, AVG(target_bm) AS target_bm, SUM(target_nmv) AS target_nmv
            FROM (
                SELECT DISTINCT date, cat, brand, margin AS target_bm, nmv AS target_nmv
                FROM performance.commercial_targets cplan
                QUALIFY CASE
                    WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
                    THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
                    ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month')
                END = DATE_TRUNC('month', date)
            ) GROUP BY ALL
        )
    ) GROUP BY ALL
)
SELECT DISTINCT bct.cat, bct.brand, bct.target_bm, ct.cat_target_margin
FROM brand_cat_target bct
LEFT JOIN cat_target ct ON ct.cat = bct.cat
'''

V2_PRODUCT_QUERY = '''
SELECT p.id AS product_id, b.name_ar AS brand, c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
WHERE p.activation = 'true'
'''

V2_GROUPS_QUERY = 'SELECT * FROM materialized_views.sku_commercial_groups_pp'

print('Supporting queries defined')

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def apply_regional_fallback(df_prices):
    """For each (product, target_region) missing from df_prices,
    borrow prices from the first available fallback region."""
    all_products = df_prices['product_id'].unique()
    fallback_rows = []

    for pid in all_products:
        pid_prices = df_prices[df_prices['product_id'] == pid]
        pid_regions = set(pid_prices['region'].unique())

        for target_region in ALL_REGIONS:
            if target_region in pid_regions:
                continue
            for fb_region in REGIONAL_FALLBACK.get(target_region, []):
                fb_data = pid_prices[pid_prices['region'] == fb_region]
                if not fb_data.empty:
                    rows = fb_data.copy()
                    rows['region'] = target_region
                    fallback_rows.append(rows)
                    break

    if fallback_rows:
        return pd.concat([df_prices] + fallback_rows, ignore_index=True)
    return df_prices


def subdivide_price_list(prices, wac, target_margin):
    """Insert intermediate prices when the margin gap between adjacent tiers
    exceeds MAX_MARGIN_GAP_PCT * target_margin."""
    if wac <= 0 or target_margin <= 0 or len(prices) < 2:
        return prices

    max_gap = MAX_MARGIN_GAP_PCT * target_margin
    result = [prices[0]]

    for i in range(1, len(prices)):
        p_lo, p_hi = result[-1], prices[i]
        if p_hi <= wac:
            result.append(p_hi)
            continue

        # If lower price is at or below WAC, start subdivision from margin=0
        m_lo = max((p_lo - wac) / p_lo, 0) if p_lo > wac else 0
        m_hi = (p_hi - wac) / p_hi
        gap = m_hi - m_lo

        if gap > max_gap:
            n_steps = int(np.ceil(gap / max_gap))
            margin_step = gap / n_steps
            for s in range(1, n_steps):
                mid_margin = m_lo + s * margin_step
                if 0 < mid_margin < 1:
                    mid_price = np.round(wac / (1 - mid_margin) * 4) / 4
                    result.append(mid_price)

        result.append(p_hi)

    return sorted(set(result))


def expand_single_price_margin_steps(price, wac, target_margin):
    """Generate tiers by stepping in margin increments centered on the single price.
    Step size = MAX_MARGIN_GAP_PCT * target_margin. 2 steps below + anchor + 2 steps above."""
    if wac <= 0 or price <= wac:
        return [price]
    
    margin_at_price = (price - wac) / price
    step = target_margin * MAX_MARGIN_GAP_PCT if target_margin > 0 else 0.03
    
    tiers = set()
    for i in [-2, -1, 0, 1, 2]:
        m = margin_at_price + i * step
        if 0 < m < 0.99:
            tiers.add(np.round(wac / (1 - m) * 4) / 4)
    tiers.add(price)  # always include anchor
    
    return sorted(tiers)


def compute_brand_price_tiers(df_all, df_wac, df_products, df_targets):
    """Compute brand-level price tiers for SKUs without direct market data.
    Groups all raw prices by (brand, cat, region), computes margin percentiles,
    and converts to price tiers using each SKU's own WAC."""
    
    existing_keys = set(zip(df_all['product_id'], df_all['region']))
    
    # df_all already has wac_p from the pipeline merge
    brand_prices = df_all[['product_id', 'region', 'price', 'wac_p']].copy()
    brand_prices = brand_prices.merge(df_products[['product_id', 'brand', 'cat']], on='product_id', how='left')
    brand_prices = brand_prices[brand_prices['wac_p'] > 0]
    brand_prices['margin'] = (brand_prices['price'] - brand_prices['wac_p']) / brand_prices['price']
    brand_prices = brand_prices[brand_prices['margin'] > 0]
    
    if len(brand_prices) == 0:
        return pd.DataFrame()
    
    brand_percs = brand_prices.groupby(['brand', 'cat', 'region']).agg(
        perc_15=('margin', lambda x: np.percentile(x, 15)),
        perc_25=('margin', lambda x: np.percentile(x, 25)),
        perc_50=('margin', lambda x: np.percentile(x, 50)),
        perc_75=('margin', lambda x: np.percentile(x, 75)),
        perc_95=('margin', lambda x: np.percentile(x, 95)),
        n_prices=('margin', 'count'),
    ).reset_index()
    brand_percs = brand_percs[brand_percs['n_prices'] >= 3]
    
    if len(brand_percs) == 0:
        return pd.DataFrame()
    
    all_products = df_products.merge(df_wac, on='product_id', how='inner')
    all_products = all_products.merge(df_targets, on=['brand', 'cat'], how='left')
    all_products['target_margin'] = all_products['target_bm'].fillna(
        all_products['cat_target_margin']).fillna(DEFAULT_TARGET_MARGIN)
    
    brand_rows = []
    for _, prod in all_products.iterrows():
        wac = prod['wac_p']
        if wac <= 0:
            continue
        for region in ALL_REGIONS:
            if (prod['product_id'], region) in existing_keys:
                continue
            bp = brand_percs[(brand_percs['brand'] == prod['brand']) & 
                            (brand_percs['cat'] == prod['cat']) &
                            (brand_percs['region'] == region)]
            if bp.empty:
                continue
            bp = bp.iloc[0]
            prices = []
            for perc_col in ['perc_15', 'perc_25', 'perc_50', 'perc_75', 'perc_95']:
                m = bp[perc_col]
                if 0 < m < 1:
                    p = np.round(wac / (1 - m) * 4) / 4
                    if p >= 0.9 * wac:
                        prices.append(p)
            if prices:
                for p in sorted(set(prices)):
                    brand_rows.append({
                        'product_id': prod['product_id'],
                        'region': region,
                        'price': p,
                        'source': 'brand_fallback',
                        'wac_p': wac,
                        'target_margin': prod['target_margin'],
                    })
    
    return pd.DataFrame(brand_rows) if brand_rows else pd.DataFrame()


def expand_to_cohorts(df_market):
    """Expand region rows to cohort_id rows for module merging."""
    rows = []
    for _, row in df_market.iterrows():
        for cid in REGION_COHORT_MAP.get(row['region'], []):
            r = row.to_dict()
            r['cohort_id'] = cid
            rows.append(r)
    return pd.DataFrame(rows)


print('Helper functions defined')

In [ ]:
# =============================================================================
# MAIN PIPELINE
# =============================================================================
def get_market_data_v2():
    """Fetch and process market prices from all sources.
    Returns one row per (product_id, region) with sorted price_tiers list."""

    print('=' * 60)
    print('MARKET DATA V2')
    print('=' * 60)

    # ---- 1. Fetch raw data ----
    print('\n1. Fetching raw data...')

    print('  1a. Ben Soliman...')
    df_ben = query_snowflake(V2_BEN_QUERY)
    print(f'      {len(df_ben)} products')

    print('  1b. Marketplace...')
    df_mp = query_snowflake(V2_MARKETPLACE_QUERY)
    print(f'      {len(df_mp)} rows')

    print('  1c. Scraped...')
    df_sc = query_snowflake(V2_SCRAPED_QUERY)
    print(f'      {len(df_sc)} rows')

    print('  1d. WAC...')
    df_wac = query_snowflake(V2_WAC_QUERY)
    print(f'      {len(df_wac)} products')

    print('  1e. Target margins...')
    df_targets = query_snowflake(V2_TARGET_MARGINS_QUERY)
    print(f'      {len(df_targets)} brand-cat targets')

    print('  1f. Product info...')
    df_products = query_snowflake(V2_PRODUCT_QUERY)
    print(f'      {len(df_products)} products')

    print('  1g. Commercial groups...')
    df_groups = query_snowflake(V2_GROUPS_QUERY)
    print(f'      {len(df_groups)} group assignments')



    # ---- 2. Expand Ben Soliman to all regions ----
    print('\n2. Expanding Ben Soliman to all regions...')
    ben_rows = []
    for _, row in df_ben.iterrows():
        for r in ALL_REGIONS:
            ben_rows.append({'product_id': row['product_id'], 'region': r,
                             'price': row['price'], 'source': 'ben_soliman'})
    df_ben_expanded = pd.DataFrame(ben_rows)
    print(f'   {len(df_ben_expanded)} rows')

    # ---- 3. Combine all sources ----
    print('\n3. Combining all sources...')
    cols = ['product_id', 'region', 'price', 'source']
    df_all = pd.concat([
        df_ben_expanded[cols],
        df_mp[cols],
        df_sc[cols],
    ], ignore_index=True)
    print(f'   {len(df_all)} total price points')

    # ---- 4. Regional fallback (marketplace + scraped only) ----
    print('\n4. Applying regional fallback...')
    df_regional = df_all[df_all['source'] != 'ben_soliman'].copy()
    df_ben_part = df_all[df_all['source'] == 'ben_soliman'].copy()
    df_regional = apply_regional_fallback(df_regional)
    df_all = pd.concat([df_ben_part, df_regional], ignore_index=True)
    print(f'   {len(df_all)} total after fallback')

    # ---- 5. WAC floor filter ----
    print('\n5. WAC floor filter (>= 0.9 * WAC)...')
    df_all = df_all.merge(df_wac, on='product_id', how='inner')
    before = len(df_all)
    df_all = df_all[df_all['price'] >= 0.9 * df_all['wac_p']].copy()
    print(f'   {before} -> {len(df_all)} (removed {before - len(df_all)})')

    # ---- 6. Target margins (brand-cat -> cat -> default 10%) ----
    print('\n6. Target margins...')
    df_all = df_all.merge(df_products[['product_id', 'brand', 'cat']], on='product_id', how='left')
    df_all = df_all.merge(df_targets, on=['brand', 'cat'], how='left')
    df_all['target_margin'] = (
        df_all['target_bm']
        .fillna(df_all['cat_target_margin'])
        .fillna(DEFAULT_TARGET_MARGIN)
    )
    df_all.drop(columns=['target_bm', 'cat_target_margin'], inplace=True, errors='ignore')
    with_brand = df_all['target_margin'].notna().sum()
    print(f'   {with_brand} rows with resolved target margin')

    # ---- 7. Deduplicate ----
    print('\n7. Deduplicating...')
    df_all['price'] = df_all['price'].round(2)
    before = len(df_all)
    df_all = df_all.drop_duplicates(subset=['product_id', 'region', 'price'])
    print(f'   {before} -> {len(df_all)}')

    # ---- 8. Brand fallback for SKUs with no market data ----
    print('\n8. Brand fallback for SKUs without market data...')
    df_brand_fb = compute_brand_price_tiers(df_all, df_wac, df_products, df_targets)
    if len(df_brand_fb) > 0:
        df_brand_fb['brand'] = None
        df_brand_fb['cat'] = None
        df_brand_fb['group_id'] = None
        df_all = pd.concat([df_all, df_brand_fb[df_all.columns.intersection(df_brand_fb.columns)]], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=['product_id', 'region', 'price'])
        print(f'   Added {len(df_brand_fb)} brand fallback prices for {df_brand_fb["product_id"].nunique()} products')
    else:
        print('   No brand fallback needed')

    # ---- 9. Commercial groups ----
    print('\n9. Commercial group price union...')
    g_cols = df_groups.columns.tolist()
    group_col = 'group_id' if 'group_id' in g_cols else g_cols[0]
    product_col = 'product_id' if 'product_id' in g_cols else g_cols[1]
    df_gc = df_groups[[group_col, product_col]].rename(
        columns={group_col: 'group_id', product_col: 'product_id'}
    ).dropna(subset=['group_id'])

    df_all = df_all.merge(df_gc, on='product_id', how='left')
    has_group = df_all['group_id'].notna()

    if has_group.any():
        df_grouped = df_all[has_group].copy()
        df_ungrouped = df_all[~has_group].copy()

        group_prices = (
            df_grouped.groupby(['group_id', 'region'])['price']
            .apply(list).reset_index()
        )
        group_prices.columns = ['group_id', 'region', 'gp_list']

        group_products = df_gc.drop_duplicates()
        gx = group_products.merge(group_prices, on='group_id', how='inner')
        gx = gx.explode('gp_list').rename(columns={'gp_list': 'price'})
        gx['price'] = gx['price'].astype(float)
        gx = gx.merge(df_wac, on='product_id', how='inner')
        gx = gx[gx['price'] >= 0.9 * gx['wac_p']]
        gx = gx.merge(df_products[['product_id', 'brand', 'cat']], on='product_id', how='left')
        gx = gx.merge(df_targets, on=['brand', 'cat'], how='left')
        gx['target_margin'] = gx['target_bm'].fillna(gx['cat_target_margin']).fillna(DEFAULT_TARGET_MARGIN)
        gx['source'] = 'group_union'
        gx = gx[['product_id', 'region', 'price', 'source', 'wac_p', 'brand', 'cat',
                  'target_margin', 'group_id']]

        df_all = pd.concat([df_ungrouped, df_grouped, gx], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=['product_id', 'region', 'price'])
        print(f'   Expanded -> {len(df_all)} total after group union + dedup')
    else:
        print('   No commercial groups found')

    # ---- 10. Source counts + market_data_source ----
    source_counts = (
        df_all.groupby(['product_id', 'region'])['source']
        .nunique().reset_index()
        .rename(columns={'source': 'num_sources'})
    )
    # Track whether SKU has direct market data or brand fallback
    source_type = df_all.groupby(['product_id', 'region'])['source'].apply(
        lambda x: 'brand' if set(x) == {'brand_fallback'} else 'sku'
    ).reset_index().rename(columns={'source': 'market_data_source'})

    # ---- 11. Aggregate into price lists ----
    print('\n10. Building price tiers...')
    agg = df_all.groupby(['product_id', 'region']).agg(
        prices=('price', lambda x: sorted(set(x))),
        wac_p=('wac_p', 'first'),
        target_margin=('target_margin', 'first'),
    ).reset_index()

    agg = agg.merge(source_counts, on=['product_id', 'region'], how='left')
    agg['num_sources'] = agg['num_sources'].fillna(1).astype(int)
    agg = agg.merge(source_type, on=['product_id', 'region'], how='left')
    agg['market_data_source'] = agg['market_data_source'].fillna('sku')

    # ---- 10b. Expand single-price SKUs (two-stage) ----
    single_mask = agg['prices'].apply(len) == 1
    single_count = single_mask.sum()
    fb_expanded = 0
    margin_expanded = 0
    
    if single_count > 0:
        for idx in agg[single_mask].index:
            pid = agg.loc[idx, 'product_id']
            region = agg.loc[idx, 'region']
            single_price = agg.loc[idx, 'prices'][0]
            wac = agg.loc[idx, 'wac_p']
            tm = agg.loc[idx, 'target_margin']
            
            # Stage 1: Check fallback regions for additional prices
            expanded = False
            for fb_region in REGIONAL_FALLBACK.get(region, []):
                fb_row = agg[(agg['product_id'] == pid) & (agg['region'] == fb_region)]
                if fb_row.empty:
                    continue
                fb_prices = fb_row.iloc[0]['prices']
                if len(fb_prices) > 1 or (len(fb_prices) == 1 and fb_prices[0] != single_price):
                    agg.at[idx, 'prices'] = sorted(set([single_price] + fb_prices))
                    expanded = True
                    fb_expanded += 1
                    break
            
            # Stage 2: Margin-step expansion centered on the single price
            if not expanded:
                agg.at[idx, 'prices'] = expand_single_price_margin_steps(single_price, wac, tm)
                margin_expanded += 1
        
        print(f'   {single_count} single-price SKUs: {fb_expanded} expanded from fallback regions, {margin_expanded} expanded with margin steps')

    # ---- 10c. Step subdivision ----
    agg['price_tiers'] = agg.apply(
        lambda row: subdivide_price_list(row['prices'], row['wac_p'], row['target_margin']),
        axis=1
    )
    agg.drop(columns=['prices'], inplace=True)

    print(f'   {len(agg)} product x region combinations')
    print(f'   Avg tiers: {agg["price_tiers"].apply(len).mean():.1f}')
    print(f'   Median tiers: {agg["price_tiers"].apply(len).median():.0f}')

    # ---- 10d. Commercial price-up induced prices ----
    print('\n11. Commercial price-up induced prices...')
    df_price_ups = get_commercial_price_ups()
    if len(df_price_ups) > 0:
        df_price_ups['ratio'] = df_price_ups['wac_p'] / df_price_ups['wac1']
        df_price_ups['new_wac_p'] = df_price_ups['new_pp'] * df_price_ups['ratio']
        induced_count = 0
        for _, pu in df_price_ups.iterrows():
            pid = pu['product_id']
            new_wac_p = pu['new_wac_p']
            if pd.isna(new_wac_p) or new_wac_p <= 0:
                continue
            mask = agg['product_id'] == pid
            if not mask.any():
                continue
            for idx in agg[mask].index:
                tm = agg.loc[idx, 'target_margin']
                if pd.isna(tm) or tm >= 1 or tm <= 0:
                    tm = 0.10
                induced = round(new_wac_p / (1 - tm) * 4) / 4
                tiers = agg.loc[idx, 'price_tiers']
                if induced > 0 and induced not in tiers:
                    agg.at[idx, 'price_tiers'] = sorted(set(tiers + [induced]))
                    induced_count += 1
        print(f'   Added induced prices to {induced_count} product-region combinations from {len(df_price_ups)} price-ups')
    else:
        print('   No commercial price-ups found')

    print('\n' + '=' * 60)
    print('MARKET DATA V2 COMPLETE')
    print('=' * 60)

    return agg[['product_id', 'region', 'price_tiers', 'wac_p', 'target_margin', 'num_sources', 'market_data_source']],df_all


print('get_market_data_v2() defined')